# 第九轮：学习何时写入记忆
训练 2–6 段，评估 2、4、6、7、8 段。比较从记忆任务中学习连续写入强度的模型，与固定变化门对照。额外报告变化时和稳定持续时的平均写入强度。仍不训练自主发声。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码版本:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Parrot')
out = root / 'trial9_results'
out.mkdir(parents=True, exist_ok=True)


## 快速检查


In [ ]:
!python trial9.py --smoke --batch 4 --out /content/Parrot/trial9_smoke


## 正式训练和评估
每 100 步将日志写入 Drive。评估报告保存每个位置的频带、时长、联合正确率及全句正确率。


In [ ]:
!python trial9.py --steps 2500 --batch 48 --out {out}


In [ ]:
import json
path = out / 'metrics.json'
report = json.loads(path.read_text(encoding='utf-8'))
report['git_commit'] = commit
path.write_text(json.dumps(report, indent=2), encoding='utf-8')
for model, groups in report.items():
    if model == 'git_commit': continue
    print(model)
    for key in ('n2_mixed_wait0','n4_mixed_wait0','n6_mixed_wait0','n8_mixed_wait0','n8_direct_wait64'):
        print(key, groups[key]['joint_all'], groups[key]['joint_per_position'])


## 回传
将 ZIP 放入项目 `result/` 目录。


In [ ]:
import shutil
archive = shutil.make_archive(str(root / 'trial9_results'), 'zip', root_dir=str(root), base_dir='trial9_results')
print(archive)
